# Suite VAL — Validators

`Validate(registry)` checks data against schemas on demand: schema validity, exact native types, links set and
filled by the right schemas, and `unique(...)` clauses. Hand-written `Fake` visitables express data proxies never
produce.

In [ ]:
import { Plain, Proxies, Schemas as S, Validators } from "@mbse/schemas/Framework";
import { NotImplementedError } from "@mbse/schemas/Framework/Errors";
import { assert, equal, Fake, raises, same, text } from "./support.js";

let Person = new S.OfObject.Builder().properties(text("name"), text("age", BigInt)).create();
let Pet = new S.OfObject.Builder().properties(text("name")).create();
const Rock = new S.OfObject.Builder().properties(text("name")).create(); // declares no adjacency
const Owns = new S.OfRelation.Builder().links("owner", "item").properties(text("since"), text("share", Number))
  .unique("since", "share").create(); // (owner, item) determine the rest
Person = new S.OfObject.Builder(Person).relations((r) => r.name("pets").of(Owns).me("owner")).update();
Pet = new S.OfObject.Builder(Pet).relations((r) => r.name("owners").of(Owns).me("item")).update();
for (const [name, schema] of [["Person", Person], ["Pet", Pet], ["Rock", Rock], ["Owns", Owns]] as const) Proxies.register(name, schema);
const B = Proxies.Builders;
const validate = Validators.Validate(B);

## VAL-01 · Natives

In [ ]:
assert(validate(new S.OfNative.Data(BigInt), 3n).length === 0);
assert(same(validate(new S.OfNative.Data(BigInt), true), ["expected int, got bool"]));
assert(same(validate(new S.OfNative.Data(Number), 1n), ["expected float, got int"]));
assert(same(validate(new S.OfNative.Data(Array), []), ["unsupported native type <class 'list'>"]));
assert(validate.OfNative(new S.OfNative.Data(String), "x").length === 0);
raises(NotImplementedError, () => validate(new S.OfUnion.Data(), 1n));

## VAL-02 · Valid data has no problems

In [ ]:
const ann = B.Person().name("Ann").age(30n).create();
const rex = B.Pet().name("Rex").create();
B.Person(ann).pets((x: any) => x.item(rex).since("2020")).update();
assert(validate(Person, ann).length === 0 && validate.OfObject(Person, ann).length === 0);
assert(validate.Reachable(Person, ann).length === 0 && validate.Reachable(Pet, rex).length === 0);
assert(validate(Person, B.Person().create()).length === 0); // nothing is mandatory

## VAL-03 · Property types, with paths

In [ ]:
{
  const bad = B.Person().name(1n).age(2.0).create();
  assert(same(validate(Person, bad), ["Person#0.name: expected str, got int", "Person#0.age: expected int, got float"]));
  assert(same(validate(Person, new Fake("Person", { colour: "red" })), ["Person#0.colour: not a property of 'Person'"]));
}

## VAL-04 · Entries: property types, unknown names, missing links, wrong schemas

In [ ]:
{
  const typed = B.Person().name("T").pets((x: any) => x.item(rex).since(2020n)).create();
  assert(same(validate(Person, typed), ["Person#0.pets[0].since: expected str, got int"]));
  const fakeRex = new Fake("Pet", { name: "fake" });
  const cases: [Fake, string][] = [
    [new Fake("Person", {}, { friends: [{}] }), "Person#0.friends: not an adjacency of 'Person'"],
    [new Fake("Person", {}, { pets: [{ item: fakeRex, colour: "red" }] }), "Person#0.pets[0].colour: not a property of the relation"],
    [new Fake("Person", {}, { pets: [{ since: "x" }] }), "Person#0.pets[0]: link 'item' is not set"],
    [new Fake("Person", {}, { pets: [{ item: new Fake("Rock", { name: "r" }) }] }), "Person#0.pets[0].item: a 'Rock' cannot fill link 'item'"],
    [new Fake("Person", {}, { pets: [{ item: new Fake("Person") }] }), "Person#0.pets[0].item: a 'Person' cannot fill link 'item'"],
  ];
  for (const [obj, message] of cases) {
    const problems = validate(Person, obj);
    assert(problems.some((p) => p.startsWith(message)), `${message} in ${problems}`);
  }
}

## VAL-05 · Linked objects are checked when reached

In [ ]:
{
  const sick = B.Pet().name(0n).create();
  B.Person(ann).pets((x: any) => x.item(sick)).update();
  assert(validate(Person, ann).length === 0); // OfObject checks only the object and its entries
  const problems = validate.Reachable(Person, ann);
  assert(problems.some((p) => p.includes("Pet#") && p.includes("name: expected str, got int")));
}

## VAL-06 · `unique(...)`: agreeing on the rest means agreeing on the clause

In [ ]:
const u1 = B.Person().name("U").create();
{
  const cat = B.Pet().name("Cat").create();
  B.Person(u1).pets((x: any) => x.item(cat).since("a")).update();
  assert(validate(Person, u1).length === 0);
  B.Person(u1).pets((x: any) => x.item(cat).since("b")).update();
  assert(same(validate(Person, u1), [
    "unique(share, since) violated: entries agreeing on ['item', 'owner'] differ on ['share', 'since']"]));
}

## VAL-07 · Uniqueness uses schema equality: absent, -0.0, NaN, and int vs bool

In [ ]:
{
  const ownerWith = (...entryProps: Record<string, unknown>[]) => {
    const o = B.Person().name("E").create();
    const pet = B.Pet().name("P").create();
    for (const props of entryProps) {
      B.Person(o).pets((x: any) => { x.item(pet); for (const [k, v] of Object.entries(props)) x[k](v); return x; }).update();
    }
    return o;
  };
  assert(validate(Person, ownerWith({}, { since: "a" })).length > 0); // absent vs present differ: a violation
  assert(validate(Person, ownerWith({ share: 0.0 }, { share: -0.0 })).length > 0); // 0.0 and -0.0 differ
  assert(validate(Person, ownerWith({ share: NaN }, { share: NaN })).length === 0); // NaNs are equal: one entry
}

## VAL-08 · Special clauses: over links only, empty, and over everything

In [ ]:
{
  const Pairs = new S.OfRelation.Builder().links("l", "r").properties(text("k")).unique("r").create(); // l and k determine r
  const Empty = new S.OfRelation.Builder().links("l", "r").properties(text("k")).unique().create();
  const All = new S.OfRelation.Builder().links("l", "r").properties(text("k")).unique("l", "r", "k").create();
  const Box = new S.OfObject.Builder().properties(text("n")).relations(
    (a) => a.name("pairs").of(Pairs).me("l"), (a) => a.name("pairs_r").of(Pairs).me("r"),
    (a) => a.name("empty").of(Empty).me("l"), (a) => a.name("empty_r").of(Empty).me("r"),
    (a) => a.name("all").of(All).me("l"), (a) => a.name("all_r").of(All).me("r")).create();
  for (const [name, schema] of [["Box", Box], ["Pairs", Pairs], ["Empty", Empty], ["All", All]] as const) Proxies.register(name, schema);
  const box = B.Box().n("box").create();
  const b1 = B.Box().n("1").create();
  const b2 = B.Box().n("2").create();
  B.Box(box).pairs((x: any) => x.r(b1).k("same")).pairs((x: any) => x.r(b2).k("same")).update();
  assert(validate(Box, box).some((p) => p.startsWith("unique(r) violated")));
  B.Box(box).empty((x: any) => x.r(b1).k("a")).empty((x: any) => x.r(b2).k("b")).update();
  assert(!validate(Box, box).some((p) => p.startsWith("unique() "))); // unique() is trivially true
  B.Box(box).all((x: any) => x.r(b1).k("a")).update();
  assert(!validate(Box, box).some((p) => p.includes("unique(k, l, r)")));
  B.Box(box).all((x: any) => x.r(b2).k("b")).update();
  assert(validate(Box, box).some((p) => p.startsWith("unique(k, l, r) violated"))); // at most one entry
}

## VAL-09 · Schema problems are reported once per schema

In [ ]:
{
  const Broken = new S.OfObject.Builder().properties(text("x", Array)).create();
  Proxies.register("Broken", Broken);
  const problems = validate.Reachable(Broken, B.Broken().create());
  assert(same(problems, ["schema 'Broken': property 'x': unsupported native type <class 'list'>"]));
}

## VAL-10 · The root must be an instance of the schema given

In [ ]:
assert(same(validate(Pet, ann), ["the value is a 'Person', not an instance of the given schema"]));

## VAL-11 · Finding F7 (pinned): uniqueness is checked only over the entries reached
`Tagged` says the tag alone determines the owner. Two components each use the tag "x"; validating one component
does not see the other, so the relation-wide violation goes unreported.

In [ ]:
{
  const Tagged = new S.OfRelation.Builder().links("owner", "thing").properties(text("tag")).unique("owner", "thing").create();
  const Holder = new S.OfObject.Builder().properties(text("n")).relations((a) => a.name("things").of(Tagged).me("owner")).create();
  const Thing = new S.OfObject.Builder().properties(text("n")).relations((a) => a.name("holders").of(Tagged).me("thing")).create();
  for (const [name, schema] of [["Tagged", Tagged], ["Holder", Holder], ["Thing", Thing]] as const) Proxies.register(name, schema);
  const h1 = B.Holder().n("h1").things((x: any) => x.thing((t: any) => t.n("t1")).tag("x")).create();
  const h2 = B.Holder().n("h2").things((x: any) => x.thing((t: any) => t.n("t2")).tag("x")).create();
  assert(validate.Reachable(Holder, h1).length === 0 && validate.Reachable(Holder, h2).length === 0);
  const h3 = B.Holder().n("h3").things((x: any) => x.thing((t: any) => t.n("t3")).tag("y"))
    .things((x: any) => x.thing((t: any) => t.n("t4")).tag("y")).create();
  assert(validate(Holder, h3).some((p) => p.startsWith("unique(owner, thing) violated"))); // within reach it is found
}

## VAL-12 · Validation reads only; it changes nothing

In [ ]:
{
  const before = Plain.ToPlain.Reachable(Person, u1); // u1 has a uniqueness violation but serializes fine
  const first = validate.Reachable(Person, u1);
  assert(same(validate.Reachable(Person, u1), first)); // repeatable
  assert(equal(Plain.ToPlain.Reachable(Person, u1), before));
  assert(ann.name === "Ann" && ann.age === 30n);
}

## VAL-13 · An embedded object's schema has no adjacencies; a value of the wrong kind is reported

In [ ]:
{
  const Wrapper = new S.OfObject.Builder().properties((p) => p.name("inner").of(Pet), text("label")).create();
  Proxies.register("Wrapper", Wrapper);
  assert(same(validate(Wrapper, new Fake("Wrapper", { inner: "x", label: "ok" })), [
    "schema 'Wrapper': property 'inner': an embedded object cannot have adjacencies",
    "Wrapper#0.inner: expected an embedded object, got str",
  ]));
}

## VAL-14 · `properties_of` reads any object's property values through the visitor protocols

In [ ]:
{
  const ann = B.Person().name("Ann").age(3n).create();
  assert(equal(Validators.properties_of(ann), new Map<string, unknown>([["name", "Ann"], ["age", 3n]])));
  assert(Validators.properties_of(B.Person().create()).size === 0); // absent properties are left out
  assert(equal(Validators.properties_of(new Fake("Person", { name: "Fay", extra: 1.5 })), new Map<string, unknown>([["name", "Fay"], ["extra", 1.5]])));
  const values = Validators.properties_of(ann);
  values.set("name", "changed");
  assert(ann.name === "Ann"); // a copy
}